# When Two Agents Take Almost the Same Actions: What Near-Mirror Episodes Reveal About Strategy

In head-to-head agent competitions, the most informative losses are not necessarily the largest. When two agents take the same actions on most turns yet finish only a few thousand coins apart, the episode begins to resemble a naturally occurring controlled comparison: most of the strategy is held constant, making the remaining divergences unusually informative.

This notebook examines the replays from two completed Kaggriculture episodes and presents a practical framework for identifying, decomposing, and interpreting these “near-mirror losses.”

## TL;DR

- Full-action agreement reached **82.9%** in Episode A and **89.9%** in Episode B. Agreement on the `farmer` component exceeded **99%** in both episodes, while agreement on the `market` component was approximately **95%** and **94%**.
- Despite following highly similar production paths, the reference player finished approximately **2.7k** and **2.3k** coins behind.
- The most consistent divergence was not a different farm plan. It was a small set of additional `SELL` orders placed during the middle and late stages of the season. Each episode contained roughly 30 such turns, and the reference player had sufficient matching inventory at all or nearly all of them.
- Valued at the pre-sell price immediately before each order, these additional `SELL` orders represented approximately **28k** and **30k** in gross sale value. Most of that apparent advantage was subsequently recovered by the shared base strategy, leaving only a **2–3k** terminal gap. The gross proxy must therefore not be interpreted as net strategy gain.

## Context & Methods

### Why near-mirror losses are unusually informative

Ordinary losses often combine many differences at once: crop mix, routing, hiring, random events, inventory, and market timing. A local action observed in such a replay may be a cause, a consequence, or irrelevant noise.

Near-mirror episodes narrow the question. If field work, `HIRE` behavior, and most market orders remain nearly identical, the analysis can focus on four questions:

1. When does the first economically meaningful divergence occur?
2. Was the divergent action feasible at that point in the episode?
3. How much of the immediate `money` difference persisted to the end?
4. Does the same pattern recur across both `player` indices, random seeds, or additional opponents?

This is not a controlled causal experiment, but it produces a more testable strategy hypothesis than the final win/loss result alone.

## Data

The analysis uses replays from two complete, 720-turn episodes, labelled **Episode A** and **Episode B**. The reference player has opposite `player` indices in the two episodes, which weakens an explanation based on one `player` index.

Throughout the notebook, `farmer`, `hands`, `market`, `money`, `HIRE`, and `SELL` retain the names used by the Kaggriculture observation and action schemas.

The analysis uses:

- the complete action submitted by each agent on every turn;
- both players’ `money`, farm states, and the shared market state;
- turn-level inventory states, used to assess whether a sale was feasible; and
- terminal inventory and farm state, used to test explanations such as unsold items at the end of the season.

### Metric definitions

**Full-action agreement**

A turn counts as a full-action agreement only when the `farmer`, `hands`, and `market` components are all identical. Component-level agreement is also reported to distinguish field-routing differences from market-layer differences.

**Money gap**

The money gap is defined as the reference player’s `money` minus the comparison player’s `money`. A negative value means the reference player is behind. The `money` field records coins in the bank; unsold items are not included in the win condition.

**Additional-`SELL` turn**

A turn is classified as an additional-`SELL` turn when the comparison player submits the same market-order prefix as the reference player and then appends one or more `SELL` orders.

**Inventory feasibility**

The check asks whether the reference player held enough of the same product immediately before the turn to place the additional order. This establishes feasibility at the divergence point; it does not assume that the subsequent state trajectory would remain unchanged.

**Gross sale-value proxy**

The proxy multiplies the pre-sell market price by the `SELL` quantity. Market-order processing, price impact, and later state feedback can all move realized outcomes away from this value. It is therefore neither profit nor a causal estimate of strategy gain.

## Results

### The two episodes are genuinely close to mirror images

| Episode | Full-action agreement | `farmer` agreement | `hands` agreement | `market` agreement | Additional-`SELL` turns | Terminal money gap |
|---|---:|---:|---:|---:|---:|---:|
| Episode A | 82.9% | 99.4% | 88.3% | 94.7% | ~28 | -2.7k |
| Episode B | 89.9% | 99.4% | 95.7% | 93.6% | ~29 | -2.3k |

The `farmer` action differs on only a handful of turns in either episode, indicating that the core route and production cadence barely change. Most full-action disagreement comes from the `hands` and `market` lists. The most stable economically meaningful pattern is a set of additional `SELL` orders appended to otherwise matching market orders.

The analysis should therefore concentrate on these small market divergences rather than attempting to re-explain the entire production plan.

### The money gap emerges during the middle and late game



*Figure 1. Money gaps remain close to zero through Day 10, widen materially after Day 20, and partially recover before the end of the season.*

The table below rounds the money gap to the nearest 0.1k. Its purpose is to show the trajectory rather than turn-level detail.

| Season anchor | Episode A | Episode B |
|---|---:|---:|
| Day 1 | 0.0k | 0.0k |
| Day 10 | 0.0k | 0.0k |
| Day 15 | -0.3k | -0.1k |
| Day 20 | -0.7k | -0.3k |
| Day 25 | -1.8k | -1.8k |
| Day 28 | -3.2k | -3.8k |
| Day 30 | -2.7k | -2.3k |

The players remain almost level for the first ten days. The gap then grows gradually, expands sharply as higher-value inventory is sold late in the season, and partially recovers before the final turn. This pattern is inconsistent with a production route that was already materially superior at the start. It is more consistent with different `SELL` timing applied to essentially the same production path.

### The key divergence is monetization timing, not a different farm plan

In both episodes, the comparison player preserves the reference player’s existing market orders on roughly 30 turns and appends additional `SELL` orders. The products involved are primarily higher-value goods such as strawberries, milk, melons, and wool.

The inventory-feasibility check produces a clear result:

- In Episode A, the reference player could support every identified additional `SELL` order with its inventory at that turn.
- In Episode B, the reference player could support the large majority of them. The few mismatches involve very low-value products and contribute little to the aggregate value.

The largest single-turn divergence occurs late in the season, when a large strawberry `SELL` order creates an immediate `money` difference of approximately **5–6k**. This is not an isolated decisive error, however. Smaller `SELL` orders recur across many turns and form the more stable pattern.

Both players finish with empty inventories. The terminal farm states are identical in Episode A; Episode B has a minor difference in weed count, but no unsold inventory or realizable production capable of explaining the terminal gap. The evidence therefore points to a narrower distinction: **both players sell all remaining inventory, but they follow different selling paths and price windows**.

### Why does 28–30k in gross sale value leave only a 2–3k terminal gap?

Conditioning the money-gap ledger on replay state reveals a pronounced “diverge, then recover” pattern:

| Episode | Money-gap change during different market actions | Recovery during later matching actions in different states | Terminal gap |
|---|---:|---:|---:|
| Episode A | ~-25k | ~+23k | -2.7k |
| Episode B | ~-24k | ~+22k | -2.3k |

An early `SELL` order changes inventory, `money`, future selling opportunities, and the shared market. Even when the players later submit identical actions, those actions operate on different inventory and `money` states. For example, the reference player may later sell the same goods through shared end-of-season selling logic, recovering much of the earlier money difference.

This decomposition is therefore a descriptive ledger, not an additive estimate of strategy value. It shows that the terminal gap forms primarily along the market-state path and is then largely recovered by the shared base strategy. Summing pre-sell price values across additional orders would materially overstate the net effect.

### Explanations that can be weakened or ruled out in these episodes

1. **An advantage tied to one `player` index.** The same pattern appears with the reference player at `player=0` and `player=1`.
2. **Failure to sell remaining inventory at the end.** Both players finish with zero inventory.
3. **Completely different farm plans.** `farmer` actions agree on more than 99% of turns, and terminal farm states are identical or very close.
4. **Early `HIRE` costs determine the outcome.** A small early hiring-cost difference exists, but its scale is far below the later money movements associated with `SELL` timing.
5. **One random incident causes the loss.** The divergence recurs across multiple sale turns and cannot be explained by a single anomalous event.

These exclusions apply only to the two observed replays. They should not be generalized into population-level causal claims.

## A Reusable Framework for Diagnosing Near-Mirror Replays

### 1. Screen candidate episodes before deep analysis

Useful candidates may satisfy several of the following conditions:

- full-action agreement above approximately 80%;
- agreement on the `market` component around 90% or higher;
- both agents complete the season without errors or early termination;
- a relatively small terminal gap, for example below roughly 5k; and
- a repeated divergence across both `player` indices or multiple replays.

These thresholds are practical screening examples, not universal cutoffs.

### 2. Decompose the action

Compare the `farmer`, `hands`, and `market` components separately. A full action can differ because of one market order appended to an otherwise identical turn; that is very different from a wholesale strategy divergence.

### 3. Locate the first economically meaningful divergence

Do not stop at the first JSON difference. Filter out movement differences with no immediate economic effect, then trace the first action that materially changes `money`, inventory, land, or productive capacity.

### 4. Test counterfactual feasibility

If an additional `SELL` order appears beneficial, verify that the reference player had sufficient inventory, that `maxMarketOrdersPerTurn` was not already binding, and that the order would not undermine later production or end-of-season selling. Without these checks, the comparison may simply reflect two different states rather than an actionable hypothesis.

### 5. Separate immediate effects from terminal effects

Record the `money` change on the divergence turn and continue tracing any recovery or amplification. In a stateful market, immediate gross value systematically overstates persistent gain.

### 6. Validate out of sample

After a near-mirror loss generates a hypothesis, test it on different random seeds, both `player` indices, and replays that were not used to discover the pattern. Two episodes can motivate an experiment; they cannot establish a general win-rate improvement.

## Questions Requiring Further Validation

The two episodes support a specific hypothesis: when the production path is highly similar, middle- and late-game `SELL` timing may determine a terminal difference of 2–3k coins.

Three questions remain:

1. Does the pattern repeat across more random seeds and both `player` indices?
2. How much of the gross sale-value proxy survives after price impact and later recovery?
3. Does earlier `SELL` timing improve terminal `money` in independent episodes once the resulting state trajectory is allowed to evolve?

## Takeaways

1. **High-similarity, small-gap losses are more useful for strategy diagnosis than low-similarity blowouts.** They reduce route and random-event confounding, making marginal differences easier to isolate.
2. **Actions that are “almost identical” can still produce different economic paths.** In a dynamic market, a small number of inventory-aware `SELL` decisions can reshape the `money` trajectory.
3. **Immediate divergence and later recovery must be analyzed together.** A gross sale-value proxy of roughly 28–30k ultimately corresponds to only a 2–3k terminal difference, demonstrating the importance of state feedback.
4. **Replay analysis should produce falsifiable hypotheses, not instant optimization conclusions.** Any candidate change still requires validation on unseen seeds and both `player` indices.

More broadly, a near-mirror loss can be treated as a natural ablation experiment in competitive agent design: the shared behavior supplies the baseline, the small divergences supply candidate explanations, and repeated validation determines whether those explanations deserve to become strategy changes.